In [12]:
from pathlib import Path
from dotenv import load_dotenv
import os
from google import genai

project_root = Path.cwd()
if not (project_root / ".env").exists():
    project_root = project_root.parent

load_dotenv(project_root / ".env", override=True)

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError(f"GEMINI_API_KEY nahi mila. .env path check karein: {project_root / '.env'}")

client = genai.Client(api_key=api_key)

In [13]:
import os
import json
import time
import re

from dotenv import load_dotenv
from google import genai
from google.genai import types

import pandas as pd
import numpy as np
import joblib
import faiss
import pickle
from sentence_transformers import SentenceTransformer

# API key is loaded from personal Gmail-linked Gemini account (college Gmail has access issues)
load_dotenv('../.env', override=True)
api_key = os.getenv('GEMINI_API_KEY')
print("API key loaded:", "Yes" if api_key else "No")

client = genai.Client(api_key=api_key)

# Models to try, in order. If the first one is overloaded (503 "high demand") or
# unavailable, call_gemini_with_retry() below automatically falls back to the next
# one -- keeps the notebook/app working during a Gemini demand spike instead of
# hard-failing on a single model name.
GEMINI_MODELS = ['gemini-3.5-flash', 'gemini-2.5-flash', 'gemini-flash-latest']


def call_gemini_with_retry(prompt, models=None, max_retries=3, base_wait=10, json_mode=False):
    """
    Calls Gemini with retry/backoff on transient errors (e.g. 503 "high demand"),
    and falls back across `models` (in order) if one is persistently unavailable.
    Returns (response_text, model_name_used). Raises the last error only if every
    model and every attempt fails.
    """
    models = models or GEMINI_MODELS
    last_error = None
    for model_name in models:
        for attempt in range(max_retries):
            try:
                config = types.GenerateContentConfig(
                    temperature=0.3,
                    response_mime_type='application/json' if json_mode else None,
                )
                response = client.models.generate_content(
                    model=model_name, contents=prompt, config=config
                )
                return response.text, model_name
            except Exception as e:
                last_error = e
                if attempt < max_retries - 1:
                    wait_time = base_wait * (attempt + 1)
                    print(f"[{model_name}] attempt {attempt + 1}/{max_retries} failed ({e}); retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"[{model_name}] failed after {max_retries} attempts; trying next model if available...")
    raise last_error


# Quick connectivity check (goes through the retry/fallback helper above, so a
# transient 503 here won't block the rest of the notebook)
test_text, used_model = call_gemini_with_retry(
    'Say "Hello, the API connection is working!" in one short sentence.'
)
print(f"\nResponse from Gemini (model used: {used_model}):")
print(test_text)


API key loaded: Yes

Response from Gemini (model used: gemini-3.5-flash):
Hello, the API connection is working!


In [14]:
# Load model and SHAP artifacts from Module 2/3
model = joblib.load('../models/churn_model.pkl')
shap_values = joblib.load('../models/shap_values.pkl')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

# Load RAG artifacts from Module 4
index = faiss.read_index('../models/policy_index.faiss')
with open('../models/policies_metadata.pkl', 'rb') as f:
    policies = pickle.load(f)
min_score = joblib.load('../models/rag_min_score.pkl')

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

churned_indices = np.where(y_test.values == 1)[0]

print("SHAP shape:", np.array(shap_values).shape, "| X_test shape:", X_test.shape)
assert np.array(shap_values).shape == X_test.shape, "SHAP shape mismatch - check class dimension (use shap_values[:, :, 1] if 3D)"
print("All artifacts loaded successfully.")
print("Number of policies in knowledge base:", len(policies))
print("Retrieval threshold:", min_score)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SHAP shape: (1409, 43) | X_test shape: (1409, 43)
All artifacts loaded successfully.
Number of policies in knowledge base: 11
Retrieval threshold: 0.4


In [15]:
def get_top_shap_reasons(customer_index, top_n=5):
    """
    Given a customer's index in X_test, return their top N churn-driving factors
    (feature name, their value, and SHAP contribution).
    """
    shap_vals = shap_values[customer_index]
    feature_vals = X_test.iloc[customer_index]

    contributions = pd.DataFrame({
        'feature': X_test.columns,
        'value': feature_vals.values,
        'shap_value': shap_vals
    })

    return contributions.sort_values('shap_value', ascending=False).head(top_n)


def build_query_from_shap(customer_index, top_n=6):
    """
    Converts a customer's top SHAP-driving features into a natural language
    query string suitable for RAG retrieval.
    """
    shap_vals = shap_values[customer_index]
    feature_vals = X_test.iloc[customer_index]

    contributions = pd.DataFrame({
        'feature': X_test.columns,
        'value': feature_vals.values,
        'shap_value': shap_vals
    })
    top_factors = contributions[contributions['shap_value'] > 0].sort_values(
        'shap_value', ascending=False
    ).head(top_n)

    query_parts = []
    contract_flags = {}

    for _, row in top_factors.iterrows():
        feature_name = row['feature']
        value = row['value']

        if feature_name.startswith('Contract_'):
            category = feature_name.split('_', 1)[1]
            contract_flags[category] = value
            continue

        if '_' in feature_name:
            base_feature, category = feature_name.rsplit('_', 1)
            if value == True:
                query_parts.append(f"{base_feature} is {category}")
            else:
                query_parts.append(f"{base_feature} is not {category}")
        else:
            query_parts.append(f"{feature_name} is {value}")

    if contract_flags:
        if not contract_flags.get('One Year', False) and not contract_flags.get('Two Year', False):
            query_parts.insert(0, "Contract is Month-to-Month")
        elif contract_flags.get('One Year', False):
            query_parts.insert(0, "Contract is One Year")
        elif contract_flags.get('Two Year', False):
            query_parts.insert(0, "Contract is Two Year")

    return "Customer profile: " + ", ".join(query_parts)


def retrieve_policy(query, top_k=3, min_score=min_score):
    """
    Given a query, retrieve the top_k most relevant policies from the knowledge base.
    Returns an empty list if no result meets the confidence threshold.
    """
    query_embedding = embed_model.encode([query])
    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        if score >= min_score:
            results.append({
                'policy_id': policies[idx]['policy_id'],
                'category': policies[idx]['category'],
                'score': float(score),   # cast numpy.float32 -> native float
                'full_text': policies[idx]['full_text']
            })

    return results


print("Helper functions defined: get_top_shap_reasons, build_query_from_shap, retrieve_policy")


Helper functions defined: get_top_shap_reasons, build_query_from_shap, retrieve_policy


In [16]:
def build_llm_prompt(customer_index, retrieved_policies, churn_probability):
    """
    Builds a grounded prompt for the LLM, including the actual predicted
    churn probability so the LLM's language stays factually consistent with
    the model's real risk assessment (not just the direction of top factors).
    """
    shap_reasons = get_top_shap_reasons(customer_index, top_n=5)

    shap_context = "\n".join([
        f"- {row['feature']}: {row['value']} (impact score: {row['shap_value']:.3f})"
        for _, row in shap_reasons.iterrows()
    ])

    if len(retrieved_policies) == 0:
        policy_context = "NO RELEVANT POLICY FOUND (score below confidence threshold)."
        has_match = False
    else:
        policy_context = "\n\n".join([
            f"[{p['policy_id']}] {p['category']} (relevance score: {p['score']:.3f})\n{p['full_text']}"
            for p in retrieved_policies
        ])
        has_match = True

    instruction = f"""Using ONLY the retrieved policy information above (if any), analyze this customer and respond in STRICT JSON format with exactly these fields:

{{
  "has_confident_match": {str(has_match).lower()},
  "summary": "One or two sentence plain-language summary. Your risk language (e.g. 'high risk', 'moderate risk', 'low risk') MUST match the actual churn probability given above ({churn_probability:.1%}) - do not call a customer high-risk if the probability is low, even if the listed factors individually look concerning.",
  "recommended_actions": ["action 1", "action 2", "action 3"],
  "policy_coverage_note": "A short honest note on whether the retrieved policy fully addresses this customer's risk factors, or only partially. If no policy matched, explain that a general check-in is recommended instead.",
  "confidence_note": "Brief note on how confident this recommendation is, based on the retrieval score"
}}

Rules:
- Do not invent actions, discounts, or numbers not present in the retrieved policy text.
- If no policy matched, recommended_actions should contain only general, low-risk steps (e.g., a check-in call), not fabricated offers.
- If the overall churn probability is low (below 40%), recommended_actions can be lighter-touch (e.g. monitoring, a soft check-in) rather than aggressive retention offers.
- Return ONLY valid JSON, no other text before or after it.
"""

    return f"""You are a customer retention assistant. A machine learning model has predicted this customer's churn probability as {churn_probability:.1%}.

CUSTOMER'S TOP CHURN-RISK FACTORS (from SHAP, i.e. factors that push toward churn, regardless of the final probability):
{shap_context}

RETRIEVED RETENTION POLICY:
{policy_context}

INSTRUCTIONS: {instruction}
"""


print("Prompt builder defined: build_llm_prompt (includes churn_probability, requests strict JSON)")


Prompt builder defined: build_llm_prompt (includes churn_probability, requests strict JSON)


In [17]:
def generate_retention_recommendation(customer_index, max_retries=5):
    """
    Full end-to-end pipeline: takes a customer index from the test set,
    runs it through SHAP explanation, RAG retrieval, and the LLM,
    and returns a grounded, structured retention recommendation.

    Uses call_gemini_with_retry() for retry/backoff + automatic model fallback
    (see GEMINI_MODELS above), and safely parses the JSON response, falling back
    gracefully if the model doesn't return valid JSON or every model/attempt fails.
    """
    customer_features = X_test.iloc[[customer_index]]
    churn_probability = float(model.predict_proba(customer_features)[0][1])

    query = build_query_from_shap(customer_index)
    retrieved_policies = retrieve_policy(query)
    prompt = build_llm_prompt(customer_index, retrieved_policies, churn_probability)

    model_used = None
    raw_text = None

    try:
        raw_text, model_used = call_gemini_with_retry(prompt, max_retries=max_retries, json_mode=True)
        cleaned = re.sub(r'^```json\s*|\s*```$', '', raw_text.strip())
        parsed_response = json.loads(cleaned)
    except json.JSONDecodeError:
        parsed_response = {
            "has_confident_match": len(retrieved_policies) > 0,
            "summary": "Could not parse structured response.",
            "recommended_actions": [],
            "policy_coverage_note": "N/A",
            "confidence_note": "N/A",
            "raw_text": raw_text
        }
    except Exception as e:
        # Template fallback: app never crashes even if every model/attempt fails
        parsed_response = {
            "has_confident_match": len(retrieved_policies) > 0,
            "summary": f"AI service unavailable. Predicted churn probability: {churn_probability:.1%}.",
            "recommended_actions": ([retrieved_policies[0]['full_text'][:200]]
                                    if retrieved_policies else ["General check-in call recommended."]),
            "policy_coverage_note": "Generated without LLM (fallback mode).",
            "confidence_note": f"LLM error: {str(e)[:100]}",
            "fallback_used": True
        }

    return {
        'customer_index': int(customer_index),                                   # numpy.int64 -> int
        'churn_probability': churn_probability,                                  # already native float
        'query_used': query,
        'retrieved_policy_ids': [p['policy_id'] for p in retrieved_policies],
        'retrieval_scores': [float(p['score']) for p in retrieved_policies],     # numpy -> float
        'model_used': model_used,                                                # which Gemini model actually answered
        'response': parsed_response
    }


print("Pipeline defined: generate_retention_recommendation (retry + automatic model fallback via call_gemini_with_retry)")


Pipeline defined: generate_retention_recommendation (retry + automatic model fallback via call_gemini_with_retry)


In [18]:
test_customer = churned_indices[0]
result = generate_retention_recommendation(test_customer)
print(json.dumps(result, indent=2))


[gemini-3.5-flash] attempt 1/5 failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}); retrying in 10s...
[gemini-3.5-flash] attempt 2/5 failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}); retrying in 20s...
[gemini-3.5-flash] attempt 3/5 failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}); retrying in 30s...
{
  "customer_index": 9,
  "churn_probability": 0.9946503043174744,
  "query_used": "Customer profile: Contract is Month-to-Month, Age is 65, Tenure in Months is 2, Total Long Distance Charges is 41.98, Payment Method is not Credit Car

In [19]:
fallback_customer = None
for idx in churned_indices[:20]:
    query_test = build_query_from_shap(idx)
    results_test = retrieve_policy(query_test)
    if len(results_test) == 0:
        fallback_customer = idx
        break

if fallback_customer is not None:
    print(f"Found a no-match case at index {fallback_customer}")
    fallback_result = generate_retention_recommendation(fallback_customer)
    print("\n--- AI-Generated Recommendation (Fallback Case) ---")
    print(json.dumps(fallback_result['response'], indent=2))
else:
    print("No fallback case found in first 20 churned customers - all had confident matches.")


Found a no-match case at index 26

--- AI-Generated Recommendation (Fallback Case) ---
{
  "has_confident_match": false,
  "summary": "This customer is at an extremely high risk of churn (100.0% probability), driven primarily by their very short tenure of one month, lack of a long-term contract, and senior age.",
  "recommended_actions": [
    "Initiate a general customer check-in call to discuss their first-month experience.",
    "Gather feedback regarding any initial setup or service difficulties they might be facing.",
    "Provide standard customer support to address any immediate usability issues."
  ],
  "policy_coverage_note": "No relevant retention policy was found to address this customer's specific risk factors. A general check-in is recommended instead of any specific promotional or contractual offers.",
  "confidence_note": "Confidence in these recommendations is low because no official retention policy was retrieved for this customer's scenario."
}


In [9]:
test_indices = [
    churned_indices[0],
    churned_indices[5],
    churned_indices[10],
]

batch_results = []
for idx in test_indices:
    print(f"Processing customer {idx}...")
    result = generate_retention_recommendation(idx)
    batch_results.append(result)
    time.sleep(3)  # small pause between calls to be gentle on the API

print("\n=== Batch Test Summary ===")
for r in batch_results:
    status = "FALLBACK (LLM failed)" if r['response'].get('fallback_used') else "SUCCESS"
    print(f"Customer {r['customer_index']} | Churn Prob: {r['churn_probability']:.1%} | Policies: {r['retrieved_policy_ids']} | Status: {status}")

print("\n=== Detail: second and third customers ===")
print(json.dumps(batch_results[1]['response'], indent=2))
print(json.dumps(batch_results[2]['response'], indent=2))


Processing customer 9...
[gemini-3.5-flash] attempt 1/5 failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}); retrying in 10s...
[gemini-3.5-flash] attempt 2/5 failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}); retrying in 20s...
Processing customer 29...
Processing customer 45...

=== Batch Test Summary ===
Customer 9 | Churn Prob: 99.5% | Policies: ['P03'] | Status: SUCCESS
Customer 29 | Churn Prob: 96.6% | Policies: ['P01C', 'P01B', 'P03'] | Status: SUCCESS
Customer 45 | Churn Prob: 9.3% | Policies: ['P03'] | Status: SUCCESS

=== Detail: second and third customers ===
{
  "has_confident_match": true,
  "summary": "This customer is at an extremely high risk of churn (96.6% probability

In [10]:
no_match_customers = []
for idx in range(len(X_test)):
    query = build_query_from_shap(idx)
    results = retrieve_policy(query)
    if len(results) == 0:
        no_match_customers.append(idx)

print(f"Found {len(no_match_customers)} customers with no confident policy match")
print("First 10 indices:", no_match_customers[:10])


Found 190 customers with no confident policy match
First 10 indices: [16, 26, 32, 37, 49, 50, 55, 62, 67, 68]


In [11]:
print("Module 5 complete. No new model artifacts to save (LLM is called live via API).")


Module 5 complete. No new model artifacts to save (LLM is called live via API).
